## Пальто мистера Белла · Пробный кадр
Гардероб частного клуба после поминок. Проверяем доступ к моделям на одном и том же описании. Это технические эскизы серии.
Общий промпт для обоих эскизов хранится в `results/pilot-cloakroom/prompt.txt`.

Для облачных ячеек установите `openai`, `python-dotenv` и `pillow`, положите учебный `.env` рядом с ноутбуком. Для ячейки Stable Diffusion нужны CUDA-версия PyTorch, `diffusers`, `transformers` и `accelerate`. Запускайте из каталога Lab1.

In [ ]:
from pathlib import Path
import base64
import io
from PIL import Image

out = Path("results/pilot-cloakroom")
out.mkdir(parents=True, exist_ok=True)
prompt = (out / "prompt.txt").read_text(encoding="utf-8").strip()

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env", override=True)
folder = os.environ["folder_id"]
client = OpenAI(base_url="https://ai.api.cloud.yandex.net/v1", project=folder,
                api_key=os.environ["api_key"], timeout=180, max_retries=0)
text_reply = client.responses.create(
    model=f"gpt://{folder}/deepseek-v4.1-flash",
    input="Назови одним словом помещение, где после приёма забирают пальто.",
)
print(text_reply.output_text)

In [ ]:
art = client.images.generate(
    model=f"art://{folder}/aliceai-image-art-3.0", prompt=prompt, n=1, size="1536x1024"
)
art_file = out / "yandex-art.png"
with Image.open(io.BytesIO(base64.b64decode(art.data[0].b64_json))) as picture:
    picture.save(art_file)
print(art_file)

In [ ]:
# Вопрос не подсказывает цвет и расположение метки.
encoded = base64.b64encode(art_file.read_bytes()).decode("ascii")
critique = client.responses.create(
    model=f"gpt://{folder}/qwen3.6-35b-a3b",
    input=[{"role": "user", "content": [
        {"type": "input_image", "image_url": f"data:image/png;base64,{encoded}"},
        {"type": "input_text", "text": "Что отличает одно из пальто? Назови видимую деталь, её цвет и расположение. Если деталь неразличима, скажи об этом."}
    ]}],
)
(out / "vlm.txt").write_text(critique.output_text, encoding="utf-8")
print(critique.output_text)

### Эскиз открытой моделью
Эта ячейка зависит только от первой ячейки с промптом и GPU. Модель Stable Diffusion 1.5 генерирует кадр 768 × 512 по тому же русскому описанию.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
local_cache = Path(".venv/hf-cache/hub")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id, revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14", variant="fp16", dtype=torch.float16, use_safetensors=True,
    cache_dir=local_cache if local_cache.is_dir() else None,
).to("cuda")

In [ ]:
image = pipe(prompt, width=768, height=512, num_inference_steps=25,
             guidance_scale=7.5, generator=torch.Generator("cuda").manual_seed(42)).images[0]
image.save(out / "stable-diffusion-1-5.png")
image

## Гардероб · диалог автора и художника
Запустите ячейки подготовки и подключения к Yandex AI Studio выше. Автор задумывает сцену, художник уточняет постановку и пишет промпт. Из их разговора получается первая версия картины. Каждый прогон сохраняется отдельно в `results/01-cloakroom/`.

In [ ]:
from datetime import datetime, timezone

brief = (
    "Первый кадр серии «Пальто мистера Белла». После поминок в гардеробе "
    "лондонского клуба остались четыре почти одинаковых пальто верблюжьего цвета. "
    "Только на манжете одного есть кривые зелёные стежки. В его подкладке зашиты "
    "деньги, но зритель пока видит лишь метку. Тон — сухая британская комедия."
)
author_model = "deepseek-v4.1-flash"
artist_model = "qwen3.6-35b-a3b"
author_role = "Ты автор серии. Отвечай по-русски, в одной-двух фразах. Выражай замысел через то, что можно увидеть."
artist_role = "Ты художник-постановщик. Пиши по-русски. Уточняй у автора, как перевести замысел в жест, деталь или композицию одного кадра."

def reply(model, instructions, history, message):
    # У каждого агента своя история. Новая реплика собеседника приходит как вход,
    # а собственный ответ остаётся в истории агента до его следующего хода.
    incoming = {"role": "user", "content": message}
    response = client.responses.create(
        model=f"gpt://{folder}/{model}", instructions=instructions,
        input=[*history, incoming],
    )
    answer = response.output_text.strip()
    if not answer:
        raise ValueError(f"Модель {model} вернула пустую реплику")
    history.extend([incoming, {"role": "assistant", "content": answer}])
    return answer

In [ ]:
def plan_scene(title, slug, scene_brief):
    author_history, artist_history = [], []
    idea = reply(author_model, author_role, author_history,
                 scene_brief + "\nПредложи одну визуальную метафору или жест для этой сцены.")
    question = reply(artist_model, artist_role, artist_history,
                     f"{scene_brief}\nАвтор предложил: {idea}\nЗадай автору один конкретный вопрос о постановке.")
    decision = reply(author_model, author_role, author_history,
                     f"Художник спрашивает: {question}\nВыбери одно решение и опиши, что увидит зритель.")
    scene_prompt = reply(artist_model, artist_role, artist_history,
                         f"Автор ответил: {decision}\nСоставь итоговый промпт для генератора. "
                         "Начни с главной детали кадра, включи решение автора. Только описание кадра, до 75 слов.")

    scene_dir = Path("results") / slug / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    scene_dir.mkdir(parents=True)
    turns = [("Автор", idea), ("Художник", question), ("Автор", decision), ("Художник", scene_prompt)]
    dialogue = "\n\n".join(f"**{role}**\n\n{text}" for role, text in turns)
    (scene_dir / "dialogue.md").write_text(
        f"# {title}\n\nАвтор — {author_model}. Художник — {artist_model}.\n\n{dialogue}\n",
        encoding="utf-8",
    )
    (scene_dir / "prompt.txt").write_text(scene_prompt + "\n", encoding="utf-8")
    (scene_dir / "бриф.txt").write_text(scene_brief + "\n", encoding="utf-8")
    return scene_dir, scene_prompt

In [ ]:
scene_dir, scene_prompt = plan_scene("Гардероб", "01-cloakroom", brief)
print((scene_dir / "dialogue.md").read_text(encoding="utf-8"))

In [ ]:
def draw_scene(description, path):
    result = client.images.generate(
        model=f"art://{folder}/aliceai-image-art-3.0", prompt=description, n=1, size="1536x1024"
    )
    with Image.open(io.BytesIO(base64.b64decode(result.data[0].b64_json))) as frame:
        frame.save(path)

In [ ]:
# Генератор получает последнюю реплику художника целиком.
draw_scene(scene_prompt, scene_dir / "01.png")
print(scene_dir / "01.png")

## Гардероб · обратная связь по изображению
Критик проверяет три видимых признака. При замечании художник переписывает промпт, затем критик рассматривает новый PNG. После двух правок выбираем по порядку приоритетов: характер штопки, четыре похожих пальто, действие гардеробщика. При равных результатах сохраняем более ранний вариант.
Для отдельного запуска задайте `scene_dir` с готовыми `01.png` и `prompt.txt`, выполните ячейки подключения, ролей и функции `draw_scene`.

In [ ]:
from pydantic import BaseModel, ConfigDict, StrictBool

class FrameReview(BaseModel):
    model_config = ConfigDict(extra="forbid")
    наблюдения: str
    улика: StrictBool
    обстановка: StrictBool
    жест: StrictBool
    правка: str

    def criteria(self):
        return (self.улика, self.обстановка, self.жест)

cloakroom_checks = ("мизерная неровная зелёная штопка на манжете одного пальто",
                    "различимы ровно четыре похожих пальто", "гардеробщик тянется за соседним пальто")

def inspect_frame(path, scene_brief, checks):
    image_data = base64.b64encode(path.read_bytes()).decode("ascii")
    task = (
        f"Бриф: {scene_brief}\nРассмотри именно присланное изображение. "
        "Верни JSON. Поле наблюдения — кратко опиши, что видно. Три логических поля: "
        f"улика — {checks[0]}; обстановка — {checks[1]}; жест — {checks[2]}. "
        "Если признак неразличим, укажи false. "
        "В поле правка предложи одну конкретную правку по первому невыполненному признаку "
        "в указанном порядке. Если всё выполнено, оставь пустую строку. Пиши по-русски."
    )
    answer = client.responses.create(
        model=f"gpt://{folder}/qwen3.6-35b-a3b",
        instructions="Ты визуальный критик. Оценивай по пикселям, опирайся на видимые детали.",
        text={"format": {"type": "json_schema", "name": "frame_review",
                         "schema": FrameReview.model_json_schema(), "strict": True}},
        input=[{"role": "user", "content": [
            {"type": "input_image", "image_url": f"data:image/png;base64,{image_data}"},
            {"type": "input_text", "text": task},
        ]}],
    )
    review = FrameReview.model_validate_json(answer.output_text)
    if not all(review.criteria()) and not review.правка.strip():
        missing = next(check for check, passed in zip(checks, review.criteria()) if not passed)
        review.правка = reply(
            artist_model, "Ты визуальный критик. Ответь по-русски одним предложением.", [],
            f"Наблюдения по кадру: {review.наблюдения}\nНе выполнен признак: {missing}. "
            "Предложи художнику одну конкретную правку композиции.",
        )
    return review

In [ ]:
def refine_scene(scene_dir, title, scene_brief, checks, run_dir=None):
    run_dir = run_dir or scene_dir / ("feedback-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    run_dir.mkdir(exist_ok=True)
    current_frame = scene_dir / "01.png"
    prompt_file = scene_dir / "prompt.txt"
    versions = []
    notes = [f"# {title} · визуальная проверка", "\n".join("- " + check for check in checks)]
    for version in range(1, 4):
        review_file = run_dir / f"{version:02}-критика.json"
        if review_file.exists():
            review = FrameReview.model_validate_json(review_file.read_text(encoding="utf-8"))
        else:
            review = inspect_frame(current_frame, scene_brief, checks)
            review_file.write_text(review.model_dump_json(indent=2), encoding="utf-8")
        versions.append((current_frame, review))
        frame_link = "../01.png" if version == 1 else current_frame.name
        prompt_link = "../prompt.txt" if version == 1 else prompt_file.name
        notes.append(f"## Версия {version}\n\n[Кадр]({frame_link}) · [Промпт]({prompt_link}) · "
                     f"[Оценка Qwen]({review_file.name})\n\n{review.наблюдения}")
        print(f"Версия {version}: {review.наблюдения}", flush=True)
        if all(review.criteria()) or version == 3:
            break
        notes.append("Предложение критика — " + review.правка)
        next_prompt = run_dir / f"{version + 1:02}-промпт.txt"
        current_frame = run_dir / f"{version + 1:02}.png"
        if next_prompt.exists():
            prompt_file = next_prompt
            if not current_frame.exists():
                draw_scene(prompt_file.read_text(encoding="utf-8").strip(), current_frame)
            continue
        revised_prompt = reply(
            artist_model, artist_role, [],
            f"Бриф: {scene_brief}\nТекущий промпт: {prompt_file.read_text(encoding='utf-8')}\n"
            f"Что увидел критик: {review.наблюдения}\nЕго правка: {review.правка}\n"
            "Исправь промпт по этой правке. Сохрани удачные детали. Верни только описание кадра по-русски.",
        )
        prompt_file = next_prompt
        prompt_file.write_text(revised_prompt + "\n", encoding="utf-8")
        draw_scene(revised_prompt, current_frame)

    best_frame, best_review = max(versions, key=lambda item: item[1].criteria())
    selected = "../01.png" if best_frame.parent == scene_dir else best_frame.name
    reason = "Все три признака подтверждены." if all(best_review.criteria()) else (
        "Достигнут предел двух правок. Выбрана лучшая версия по приоритету улики, обстановки и жеста."
    )
    notes.append(f"## Выбор\n\n[Выбранный кадр]({selected}). {reason}\n\n{best_review.наблюдения}")
    (run_dir / "сравнение.md").write_text("\n\n".join(notes) + "\n", encoding="utf-8")
    return best_frame

In [ ]:
selected_frame = refine_scene(scene_dir, "Гардероб", brief, cloakroom_checks)
print(selected_frame)

## Два генератора · один гардероб
Для парного сравнения берём два признака сцены: ряд из четырёх одинаково окрашенных пальто и единственную зелёную штопку. Короткая формулировка помещается в текстовый энкодер Stable Diffusion. Это два варианта одного сюжета. Для запуска нужны ячейки подготовки, подключения, загрузки Stable Diffusion и определения `draw_scene`.

In [ ]:
comparison = Path("results/01-cloakroom/model-comparison")
comparison_prompt = (comparison / "prompt.txt").read_text(encoding="utf-8").strip()
tokens = pipe.tokenizer(comparison_prompt, truncation=False).input_ids
if len(tokens) > pipe.tokenizer.model_max_length:
    raise ValueError("Промпт не помещается в текстовый энкодер Stable Diffusion")

draw_scene(comparison_prompt, comparison / "yandex-art.png")
sd_image = pipe(
    comparison_prompt, width=768, height=512, num_inference_steps=25, guidance_scale=7.5,
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]
sd_image.save(comparison / "stable-diffusion-1-5.png")
print(f"Длина промпта со служебными токенами: {len(tokens)} из {pipe.tokenizer.model_max_length}")

## Пальто мистера Белла · три продолжения
В кэбе зритель узнаёт пальто по стежкам. В ломбарде мужчины вскрывают подкладку другого пальто с целой манжетой. В примерочной зелёную нитку находит покупательница. У каждой сцены свой бриф и три визуальные проверки.

In [ ]:
series_style = (
    "Серия «Пальто мистера Белла», сухая британская криминальная комедия без надписей. "
    "Кинофотография: шерсть верблюжьего цвета, тёмные фактуры, боковой свет. "
)
episodes = [
    ("Заднее сиденье", "02-cab",
     "На заднем кожаном сиденье чёрного лондонского кэба забыли бежевое пальто. "
     "У самого края рукава видна кривая починка зелёной ниткой. Через заднее стекло "
     "видна уходящая фигура пассажира. В зеркале водитель замечает пальто.",
     ("зелёные стежки на манжете лежащего бежевого пальто",
      "салон такси с задним сиденьем", "взгляд водителя в зеркале или уходящий пассажир")),
    ("Пустая подкладка", "03-pawnshop",
     "В тесном ломбарде в Сохо трое мужчин в дорогих костюмах разглядывают изнанку "
     "бежевого пальто на прилавке. Подкладка распорота и пуста. На виду гладкая манжета "
     "без зелёной нитки: они нашли другое пальто. На полках подержанные часы и украшения.",
     ("вывернутая пустая подкладка бежевого пальто и видимая однотонная манжета",
      "прилавок ломбарда с подержанными ценностями", "мужчины смотрят внутрь подкладки")),
    ("Скидка за штопку", "04-fitting-room",
     "Примерочная благотворительного магазина. Пенсионерка надела бежевое пальто "
     "и отворачивает манжету с несколькими кривыми зелёными стежками. Она указывает "
     "продавцу на починку и протягивает ценник, добиваясь скидки. Рядом зеркало и дешёвые вешалки.",
     ("на манжете бежевого пальто видна зелёная штопка", "примерочная с зеркалом и вешалками",
      "пожилая покупательница показывает продавцу починку или ценник")),
]

In [ ]:
def paint_episode(episode):
    title, slug, synopsis, checks = episode
    scene_brief = series_style + synopsis
    directory, description = plan_scene(title, slug, scene_brief)
    draw_scene(description, directory / "01.png")
    selected = refine_scene(directory, title, scene_brief, checks)
    print(title, selected, flush=True)
    return selected

In [ ]:
series_frames = [paint_episode(episode) for episode in episodes]